# Plant Disease Domain Gap: Part 2, Improving Field Performance

This notebook continues `Plant_Disease_Domain_Gap_Experiments.ipynb`. It rebuilds the same audited splits (cells 1 to 3 are identical), then tries to raise PlantDoc (field) performance while keeping PlantVillage (lab) performance high. All model choices use validation data only. PlantDoc test is evaluated once per selected checkpoint.

Team: Muskan Ejaz, Juwairiya Haroon, Zainab Hashmi.

In [1]:
import os, sys, time, json, copy, random
from pathlib import Path
from collections import Counter, defaultdict
from hashlib import sha256

import numpy as np
import pandas as pd
from PIL import Image

IN_COLAB = "google.colab" in sys.modules
ROOT = Path("/content") if IN_COLAB else Path(os.environ.get("PDG_ROOT", Path.cwd().parent / "data")).resolve()
OUT = (Path("/content/outputs/part2") if IN_COLAB else Path.cwd() / "outputs" / "part2")
OUT.mkdir(parents=True, exist_ok=True)
SMOKE = os.environ.get("PDG_SMOKE") == "1"   # quick code check on a small subset
print("Data folder:", ROOT)
print("Output folder:", OUT)
print("Smoke test:", SMOKE)

Data folder: ~/Desktop/Plant_Disease_Domain_Gap/data
Output folder: ~/Desktop/Plant_Disease_Domain_Gap/repo/outputs/part2
Smoke test: False


## 1. Data download (skipped if the files already exist)

In [2]:
import subprocess
from huggingface_hub import hf_hub_download

plantdoc_path = ROOT / "PlantDoc"
if not (plantdoc_path / "test").is_dir():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/pratikkayal/PlantDoc-Dataset.git", str(plantdoc_path)], check=True)

pv_files = {}
for name in ("splits/color_train.txt", "splits/color_test.txt", "data.zip"):
    pv_files[name] = hf_hub_download(repo_id="mohanty/PlantVillage", repo_type="dataset",
                                     filename=name, local_dir=ROOT / "PlantVillage_hf")
print("PlantDoc and PlantVillage files are available.")

~/Desktop/cv/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PlantDoc and PlantVillage files are available.


## 2. Same class mapping, pilot selection and audit as the baseline notebook

In [3]:
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def list_images(folder):
    return sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in image_extensions)

plantdoc_counts = pd.DataFrame({
    split: {f.name: len(list_images(f)) for f in (plantdoc_path / split).iterdir()
            if f.is_dir() and not f.name.startswith(".")}
    for split in ("train", "test")
}).fillna(0).astype(int).sort_index()

color_lists = {split: [l.strip() for l in Path(pv_files[f"splits/color_{split}.txt"]).read_text().splitlines() if l.strip()]
               for split in ("train", "test")}

class_map = {
    "Apple Scab Leaf": "Apple___Apple_scab",
    "Corn Gray leaf spot": "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "Potato leaf early blight": "Potato___Early_blight",
    "Potato leaf late blight": "Potato___Late_blight",
    "Squash Powdery mildew leaf": "Squash___Powdery_mildew",
    "grape leaf black rot": "Grape___Black_rot",
    "Tomato Early blight leaf": "Tomato___Early_blight",
    "Tomato Septoria leaf spot": "Tomato___Septoria_leaf_spot",
    "Tomato leaf bacterial spot": "Tomato___Bacterial_spot",
    "Tomato leaf late blight": "Tomato___Late_blight",
    "Tomato leaf mosaic virus": "Tomato___Tomato_mosaic_virus",
    "Tomato mold leaf": "Tomato___Leaf_Mold",
}
pv_counts = {split: Counter(Path(p).parent.name for p in paths) for split, paths in color_lists.items()}
for doc_label, village_label in class_map.items():
    assert plantdoc_counts.loc[doc_label, "test"] > 0
    assert pv_counts["train"][village_label] > 0 and pv_counts["test"][village_label] > 0

mapping_table = pd.DataFrame({
    "PlantVillage class": pd.Series(class_map),
    "PlantVillage train list": [pv_counts["train"][v] for v in class_map.values()],
    "PlantVillage test": [pv_counts["test"][v] for v in class_map.values()],
    "PlantDoc train": [plantdoc_counts.loc[d, "train"] for d in class_map],
    "PlantDoc test": [plantdoc_counts.loc[d, "test"] for d in class_map],
})
mapping_table.index.name = "PlantDoc class"
mapping_table.to_csv(OUT / "class_mapping_counts.csv")
display(mapping_table)
print("PlantDoc test images in the pilot:", mapping_table["PlantDoc test"].sum())

,PlantVillage class,PlantVillage train list,PlantVillage test,PlantDoc train,PlantDoc test
PlantDoc class,,,,,
Apple Scab Leaf,Apple___Apple_scab,503,127,83,10
Corn Gray leaf spot,Corn_(maize)___Cercospora_leaf_spot Gray_leaf_...,406,107,64,4
Potato leaf early blight,Potato___Early_blight,836,164,109,8
Potato leaf late blight,Potato___Late_blight,788,212,97,8
Squash Powdery mildew leaf,Squash___Powdery_mildew,1457,378,124,6
grape leaf black rot,Grape___Black_rot,1012,168,56,8
Tomato Early blight leaf,Tomato___Early_blight,785,215,79,9
Tomato Septoria leaf spot,Tomato___Septoria_leaf_spot,1434,337,140,11
Tomato leaf bacterial spot,Tomato___Bacterial_spot,1691,436,101,9


PlantDoc test images in the pilot: 99


In [4]:
from zipfile import ZipFile

pilot_classes = set(class_map.values())
pilot_paths = {split: [p for p in paths if Path(p).parent.name in pilot_classes]
               for split, paths in color_lists.items()}
pv_image_root = ROOT / "PlantVillage_pilot"

if not (pv_image_root / "done.txt").exists():
    with ZipFile(pv_files["data.zip"]) as archive:
        color_members = {m[m.index("raw/color/"):]: m for m in archive.namelist()
                         if "raw/color/" in m and not m.endswith("/")}
        for split, paths in pilot_paths.items():
            for path in paths:
                rel = path.removeprefix("./")
                target = pv_image_root / split / rel
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(archive.read(color_members[rel]))
    (pv_image_root / "done.txt").write_text("ok")
for split in pilot_paths:
    print(f"PlantVillage {split}: {len(pilot_paths[split]):,} pilot images")

PlantVillage train: 11,533 pilot images
PlantVillage test: 2,757 pilot images


In [5]:
def audit_images(paths):
    hashes, unreadable = defaultdict(list), []
    for path in paths:
        try:
            with Image.open(path) as image:
                image.load()
            hashes[sha256(path.read_bytes()).hexdigest()].append(path)
        except (OSError, ValueError):
            unreadable.append(path)
    return hashes, unreadable

audit_paths = {
    "PlantVillage train": [pv_image_root / "train" / p for p in pilot_paths["train"]],
    "PlantVillage test": [pv_image_root / "test" / p for p in pilot_paths["test"]],
    "PlantDoc test": [p for label in class_map for p in list_images(plantdoc_path / "test" / label)],
    "PlantDoc train": [p for label in class_map for p in list_images(plantdoc_path / "train" / label)],
}
image_audits = {}
for name, paths in audit_paths.items():
    image_audits[name] = audit_images(paths)
    print(f"{name}: {len(paths):,} files, {len(image_audits[name][1])} unreadable")

PlantVillage train: 11,533 files, 0 unreadable


PlantVillage test: 2,757 files, 0 unreadable


PlantDoc test: 99 files, 0 unreadable


PlantDoc train: 1,083 files, 0 unreadable


In [6]:
source_hashes = image_audits["PlantVillage train"][0]
lab_test_hashes = set(image_audits["PlantVillage test"][0])
field_test_hashes = set(image_audits["PlantDoc test"][0])
test_hashes = lab_test_hashes | field_test_hashes
print("PlantVillage test hashes shared with train:", len(source_hashes.keys() & lab_test_hashes))
print("PlantDoc test hashes shared with train:", len(source_hashes.keys() & field_test_hashes))

clean_source_hashes = {h: p for h, p in source_hashes.items() if h not in test_hashes}
clean_count = sum(len(p) for p in clean_source_hashes.values())
source_rows = []
for fingerprint, paths in sorted(clean_source_hashes.items()):
    labels = {p.parent.name for p in paths}
    assert len(labels) == 1
    source_rows.append({"path": min(paths), "label": labels.pop(), "hash": fingerprint})
source_data = pd.DataFrame(source_rows)
print("Training files excluded (match a test image):", len(audit_paths["PlantVillage train"]) - clean_count)
print("Extra copies excluded inside the source pool:", clean_count - len(source_data))

from sklearn.model_selection import train_test_split
train_data, val_data = train_test_split(source_data, test_size=0.15, random_state=42,
                                        stratify=source_data["label"])
print("Training images:", len(train_data), " Validation images:", len(val_data))
assert SMOKE or (len(train_data), len(val_data)) == (9796, 1729), "Split differs from the baseline notebook"

PlantVillage test hashes shared with train: 3
PlantDoc test hashes shared with train: 0
Training files excluded (match a test image): 3
Extra copies excluded inside the source pool: 5


Training images: 9796  Validation images: 1729


## 3. Audit of PlantDoc training images before field adaptation

SHA-256 only finds byte-identical files. We also compute a 64-bit difference hash (dHash) of every PlantDoc image. Any PlantDoc training image within Hamming distance 6 of a PlantDoc test image is treated as a near duplicate and left out of adaptation, so the field test set stays unseen.

In [7]:
def dhash(path, size=8):
    with Image.open(path) as im:
        g = im.convert("L").resize((size + 1, size), Image.BILINEAR)
    px = np.asarray(g, dtype=np.int16)
    bits = (px[:, 1:] > px[:, :-1]).flatten()
    return int("".join("1" if b else "0" for b in bits), 2)

hamming = lambda a, b: bin(a ^ b).count("1")
NEAR_DUP_THRESHOLD = 6

pd_test_paths = audit_paths["PlantDoc test"]
pd_train_hashes = image_audits["PlantDoc train"][0]
test_dh = {p: dhash(p) for p in pd_test_paths}

field_rows, near_pairs, exact_pairs = [], [], 0
seen = set()
for fingerprint, paths in sorted(pd_train_hashes.items()):
    path = min(paths)
    labels = {p.parent.name for p in paths}
    if fingerprint in test_hashes or fingerprint in source_hashes:
        exact_pairs += 1
        continue
    if len(labels) != 1:
        continue
    d = dhash(path)
    close = [(t, hamming(d, td)) for t, td in test_dh.items() if hamming(d, td) <= NEAR_DUP_THRESHOLD]
    if close:
        near_pairs += [(path, t, dist) for t, dist in close]
        continue
    field_rows.append({"path": path, "label": class_map[labels.pop()], "hash": fingerprint})

field_pool = pd.DataFrame(field_rows)
n_train_files = len(audit_paths["PlantDoc train"])
n_unique = len(pd_train_hashes)
n_near_files = len({p for p, _, _ in near_pairs})
print(f"PlantDoc train files in the 12 classes: {n_train_files}")
print(f"Exact copies inside PlantDoc train removed: {n_train_files - n_unique - len(image_audits['PlantDoc train'][1])}")
print(f"Byte-identical to a test or source image: {exact_pairs}")
print(f"Near duplicates of a PlantDoc test image (dHash <= {NEAR_DUP_THRESHOLD}): {n_near_files} files, {len(near_pairs)} pairs")
print(f"Field pool kept for adaptation: {len(field_pool)}")

field_adapt, field_val = train_test_split(field_pool, test_size=0.2, random_state=42, stratify=field_pool["label"])
print("Field adaptation images:", len(field_adapt), " Field validation images:", len(field_val))
pd.DataFrame(near_pairs, columns=["plantdoc_train", "plantdoc_test", "dhash_distance"]).to_csv(OUT / "near_duplicates.csv", index=False)

PlantDoc train files in the 12 classes: 1083
Exact copies inside PlantDoc train removed: 1
Byte-identical to a test or source image: 6
Near duplicates of a PlantDoc test image (dHash <= 6): 4 files, 4 pairs
Field pool kept for adaptation: 1071
Field adaptation images: 856  Field validation images: 215


## 4. Improving field performance while keeping lab performance

Part 1 showed the size of the gap with a frozen ResNet-18. Part 2 tries to close it with two changes:

* **Backbone comparison (P1).** Five ImageNet backbones are used as frozen feature extractors. For each one, a linear head is trained on (a) PlantVillage only and (b) PlantVillage plus the 856 field adaptation images, with the two domains weighted equally. The epoch is chosen on validation data only.
* **Joint fine-tuning (P2).** The backbone with the best mean validation macro-F1 (PlantVillage val and field val) is fine-tuned on both domains. The last stages are unfrozen, the field-style augmentation from Part 1 is used, and every epoch mixes all field adaptation images with twice as many randomly drawn PlantVillage images. The epoch with the best mean validation macro-F1 is kept. Test-time augmentation (TTA) averages the predictions for the image and its horizontal flip.

PlantDoc test is used only for the final evaluation of each selected checkpoint.

In [8]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import torchvision.transforms as T
from torchvision import models
from sklearn.metrics import f1_score, accuracy_score, precision_recall_fscore_support

fast = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
NUM_WORKERS = 2 if IN_COLAB else 0
class_names = sorted(class_map.values())
label_to_id = {label: i for i, label in enumerate(class_names)}
K = len(class_names)

def resized_cache(paths, short=256):
    cache = {}
    for p in paths:
        with Image.open(p) as im:
            im = im.convert("RGB"); w, h = im.size; s = short / min(w, h)
            cache[p] = im.resize((max(short, round(w * s)), max(short, round(h * s))), Image.BILINEAR) if min(w, h) > short else im.copy()
    return cache
field_cache = resized_cache(list(field_adapt["path"]) + list(field_val["path"]) + list(pd_test_paths))

class LeafDataset(Dataset):
    def __init__(self, rows, transform, cache=field_cache):
        self.samples = list(zip(rows["path"], rows["label"])); self.transform, self.cache = transform, cache
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        path, label = self.samples[i]
        if path in self.cache: image = self.cache[path]
        else:
            with Image.open(path) as im: image = im.convert("RGB")
        return self.transform(image), label_to_id[label]

test_frames = {
    "PlantVillage test": pd.DataFrame({"path": sorted(audit_paths["PlantVillage test"]),
                                       "label": [p.parent.name for p in sorted(audit_paths["PlantVillage test"])]}),
    "PlantDoc test": pd.DataFrame({"path": sorted(pd_test_paths), "label": [class_map[p.parent.name] for p in sorted(pd_test_paths)]}),
}
if SMOKE:
    train_data = train_data.groupby("label").head(12); val_data = val_data.groupby("label").head(6)
    field_adapt = field_adapt.groupby("label").head(6); field_val = field_val.groupby("label").head(3)
    test_frames["PlantVillage test"] = test_frames["PlantVillage test"].groupby("label").head(10)
frames = {"source_train": train_data, "source_val": val_data, "field_adapt": field_adapt, "field_val": field_val,
          "pv_test": test_frames["PlantVillage test"], "pd_test": test_frames["PlantDoc test"]}
print({k: len(v) for k, v in frames.items()}, "| device:", fast)

def macro_f1(y, pred):
    return 100 * f1_score(y, pred, labels=range(K), average="macro", zero_division=0)

def bootstrap_ci(y, pred, n=2000, seed=0):
    rng = np.random.default_rng(seed); y, pred = np.asarray(y), np.asarray(pred)
    vals = [macro_f1(y[i], pred[i]) for i in (rng.integers(0, len(y), len(y)) for _ in range(n))]
    return np.percentile(vals, [2.5, 97.5])

{'source_train': 9796, 'source_val': 1729, 'field_adapt': 856, 'field_val': 215, 'pv_test': 2757, 'pd_test': 99} | device: mps


### P1. Backbone comparison with frozen features

In [9]:
BACKBONES = {
    "ResNet-18": (models.resnet18, models.ResNet18_Weights.IMAGENET1K_V1),
    "ResNet-50": (models.resnet50, models.ResNet50_Weights.IMAGENET1K_V2),
    "EfficientNet-B0": (models.efficientnet_b0, models.EfficientNet_B0_Weights.IMAGENET1K_V1),
    "ConvNeXt-Tiny": (models.convnext_tiny, models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1),
    "Swin-T": (models.swin_t, models.Swin_T_Weights.IMAGENET1K_V1),
}

def build(name, num_classes=None):
    fn, w = BACKBONES[name]; m = fn(weights=w)
    if name.startswith("ResNet"):
        dim = m.fc.in_features; m.fc = nn.Linear(dim, num_classes) if num_classes else nn.Identity()
    elif name.startswith("EfficientNet"):
        dim = m.classifier[1].in_features; m.classifier[1] = nn.Linear(dim, num_classes) if num_classes else nn.Identity()
    elif name.startswith("ConvNeXt"):
        dim = m.classifier[2].in_features; m.classifier[2] = nn.Linear(dim, num_classes) if num_classes else nn.Identity()
    else:
        dim = m.head.in_features; m.head = nn.Linear(dim, num_classes) if num_classes else nn.Identity()
    return m, dim, w.transforms()

@torch.inference_mode()
def extract(net, tf, rows):
    feats, ys = [], []
    for x, y in DataLoader(LeafDataset(rows, tf), batch_size=64, num_workers=NUM_WORKERS):
        feats.append(net(x.to(fast)).float().cpu()); ys.append(y)
    return torch.cat(feats), torch.cat(ys)

def train_probe(F, use_field, epochs=30, seed=0, lr=1e-3, batch=64):
    """Linear head on frozen features. With use_field, field images get the same total weight as PlantVillage."""
    torch.manual_seed(seed); g = torch.Generator().manual_seed(seed)
    Xs, ys = F["source_train"]
    if use_field:
        Xf, yf = F["field_adapt"]
        X, y = torch.cat([Xs, Xf]), torch.cat([ys, yf])
        w = torch.cat([torch.full((len(ys),), 0.5 / len(ys)), torch.full((len(yf),), 0.5 / len(yf))]) * len(y)
    else:
        X, y, w = Xs, ys, torch.ones(len(ys))
    head = nn.Linear(X.shape[1], K); opt = torch.optim.Adam(head.parameters(), lr=lr, weight_decay=1e-5)
    ce = nn.CrossEntropyLoss(reduction="none"); best, log = (-1, None, 0), []
    for ep in range(1, epochs + 1):
        perm = torch.randperm(len(y), generator=g)
        for i in range(0, len(y), batch):
            idx = perm[i:i + batch]; opt.zero_grad()
            loss = (ce(head(X[idx]), y[idx]) * w[idx]).mean(); loss.backward(); opt.step()
        with torch.no_grad():
            pv = macro_f1(F["source_val"][1], head(F["source_val"][0]).argmax(1))
            fv = macro_f1(F["field_val"][1], head(F["field_val"][0]).argmax(1))
        score = (pv + fv) / 2 if use_field else pv          # source-only probes never look at field data
        log.append((ep, pv, fv))
        if score > best[0]: best = (score, copy.deepcopy(head.state_dict()), ep)
    head.load_state_dict(best[1]); return head, best[2], log

probe_rows, probe_preds = [], {}
for name in BACKBONES:
    t0 = time.time()
    net, dim, tf = build(name); net = net.to(fast).eval()
    F = {k: extract(net, tf, v) for k, v in frames.items()}
    del net
    for use_field in (False, True):
        head, ep, log = train_probe(F, use_field, epochs=3 if SMOKE else 30)
        with torch.no_grad():
            out = {k: head(F[k][0]).argmax(1).numpy() for k in ("source_val", "field_val", "pv_test", "pd_test")}
        row_ = {"backbone": name, "training data": "PV + field" if use_field else "PV only", "feature dim": dim, "epoch": ep,
                "PV val F1": macro_f1(F["source_val"][1], out["source_val"]), "field val F1": macro_f1(F["field_val"][1], out["field_val"]),
                "PV test acc": 100 * accuracy_score(F["pv_test"][1], out["pv_test"]), "PV test F1": macro_f1(F["pv_test"][1], out["pv_test"]),
                "PD test acc": 100 * accuracy_score(F["pd_test"][1], out["pd_test"]), "PD test F1": macro_f1(F["pd_test"][1], out["pd_test"])}
        probe_rows.append(row_); probe_preds[(name, row_["training data"])] = out["pd_test"]
    print(f"{name}: done in {time.time() - t0:.0f}s", flush=True)
    if fast.type == "mps": torch.mps.empty_cache()

probes = pd.DataFrame(probe_rows)
probes["mean val F1"] = (probes["PV val F1"] + probes["field val F1"]) / 2
probes.to_csv(OUT / "p1_backbone_probes.csv", index=False)
display(probes.round(2))

ResNet-18: done in 49s


ResNet-50: done in 161s


EfficientNet-B0: done in 97s


ConvNeXt-Tiny: done in 195s


Swin-T: done in 242s


,backbone,training data,feature dim,epoch,PV val F1,field val F1,PV test acc,PV test F1,PD test acc,PD test F1,mean val F1
0,ResNet-18,PV only,512,29,94.61,27.25,95.36,95.52,32.32,31.18,60.93
1,ResNet-18,PV + field,512,21,88.10,60.11,89.41,89.02,50.51,53.69,74.10
2,ResNet-50,PV only,2048,30,98.03,35.29,97.50,97.59,30.30,28.90,66.66
3,ResNet-50,PV + field,2048,22,94.97,57.44,95.36,95.39,43.43,44.87,76.20
4,EfficientNet-B0,PV only,1280,29,98.61,32.28,96.99,97.30,39.39,39.61,65.44
5,EfficientNet-B0,PV + field,1280,28,95.29,62.02,95.36,95.36,56.57,57.45,78.66
6,ConvNeXt-Tiny,PV only,768,23,98.49,37.28,98.08,98.24,34.34,32.16,67.89
7,ConvNeXt-Tiny,PV + field,768,14,94.61,66.14,95.57,95.60,58.59,62.10,80.38
8,Swin-T,PV only,768,20,98.61,40.10,98.15,98.23,36.36,34.86,69.35
9,Swin-T,PV + field,768,19,95.35,65.43,95.61,95.76,52.53,51.68,80.39


In [10]:
joint = probes[probes["training data"] == "PV + field"]
best_backbone = joint.loc[joint["mean val F1"].idxmax(), "backbone"]
print("Backbone selected by mean validation macro-F1:", best_backbone)

Backbone selected by mean validation macro-F1: Swin-T


### P2. Joint fine-tuning of the selected backbone

In [11]:
augment_tf = lambda tf: T.Compose([
    T.RandomResizedCrop(224, scale=(0.35, 1.0)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(25),
    T.ColorJitter(0.4, 0.4, 0.4, 0.05), T.RandomApply([T.GaussianBlur(5, sigma=(0.1, 2.0))], p=0.3),
    T.ToTensor(), T.Normalize(tf.mean, tf.std)])

def trainable_parts(name, m):
    """Last stages + classifier are fine-tuned; earlier stages stay frozen."""
    if name.startswith("ResNet"): return [m.layer3, m.layer4, m.fc]
    if name.startswith("EfficientNet"): return [m.features[6], m.features[7], m.features[8], m.classifier]
    if name.startswith("ConvNeXt"): return [m.features[5], m.features[6], m.features[7], m.classifier]
    return [m.features[5], m.features[6], m.features[7], m.norm, m.head]

def set_train_mode(m):
    m.train()
    for mod in m.modules():                      # batch-norm statistics stay frozen (small field set)
        if isinstance(mod, nn.BatchNorm2d): mod.eval()

@torch.inference_mode()
def predict(m, tf, rows, tta=False):
    m.eval(); logits, ys = [], []
    for x, y in DataLoader(LeafDataset(rows, tf), batch_size=64, num_workers=NUM_WORKERS):
        x = x.to(fast); p = m(x).softmax(1)
        if tta: p = (p + m(torch.flip(x, dims=[3])).softmax(1)) / 2
        logits.append(p.float().cpu()); ys.append(y)
    return torch.cat(logits), torch.cat(ys)

def val_f1(m, tf, rows):
    prob, y = predict(m, tf, rows)
    return macro_f1(y, prob.argmax(1))

def joint_finetune(name, epochs, seed, pv_per_epoch_ratio=2):
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    m, dim, tf = build(name, K); m.requires_grad_(False)
    parts = trainable_parts(name, m)
    head_params = list(parts[-1].parameters()); body_params = [p for part in parts[:-1] for p in part.parameters()]
    for p in head_params + body_params: p.requires_grad_(True)
    m = m.to(fast)
    opt = torch.optim.AdamW([{"params": body_params, "lr": 1e-4}, {"params": head_params, "lr": 1e-3}], weight_decay=0.05)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    aug = augment_tf(tf); ce = nn.CrossEntropyLoss(label_smoothing=0.1)
    rng = np.random.default_rng(seed); log, best = [], (-1, None, 0)
    for ep in range(1, epochs + 1):
        pv_rows = train_data.iloc[rng.choice(len(train_data), min(len(train_data), pv_per_epoch_ratio * len(field_adapt)), replace=False)]
        loader = DataLoader(ConcatDataset([LeafDataset(pv_rows, aug), LeafDataset(field_adapt, aug)]), batch_size=32,
                            shuffle=True, num_workers=NUM_WORKERS, generator=torch.Generator().manual_seed(seed * 100 + ep))
        set_train_mode(m); total, n = 0.0, 0
        for x, y in loader:
            opt.zero_grad(); loss = ce(m(x.to(fast)), y.to(fast)); loss.backward(); opt.step()
            total += loss.item() * len(y); n += len(y)
        sched.step()
        pv, fv = val_f1(m, tf, val_data), val_f1(m, tf, field_val)
        log.append({"epoch": ep, "train_loss": total / n, "pv_val_f1": pv, "field_val_f1": fv, "mean_val_f1": (pv + fv) / 2})
        print(f"  {name} seed {seed} epoch {ep}: loss {total / n:.3f} | PV val {pv:.2f} | field val {fv:.2f}", flush=True)
        if (pv + fv) / 2 > best[0]: best = ((pv + fv) / 2, {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}, ep)
    m.load_state_dict(best[1]); return m, tf, pd.DataFrame(log), best[2]

P2_EPOCHS = 2 if SMOKE else 12
P2_SEEDS = [0] if SMOKE else [0, 1, 2]
p2_rows, p2_logs, p2_preds = [], {}, {}
for seed in P2_SEEDS:
    t0 = time.time()
    m, tf, log, ep = joint_finetune(best_backbone, P2_EPOCHS, seed)
    p2_logs[seed] = log; log.to_csv(OUT / f"p2_epoch_log_seed{seed}.csv", index=False)
    for tta in (False, True):
        pvp, pvy = predict(m, tf, test_frames["PlantVillage test"], tta)
        pdp, pdy = predict(m, tf, test_frames["PlantDoc test"], tta)
        r = {"backbone": best_backbone, "seed": seed, "TTA": tta, "epoch": ep,
             "PV val F1": log.loc[ep - 1, "pv_val_f1"], "field val F1": log.loc[ep - 1, "field_val_f1"],
             "PV test acc": 100 * accuracy_score(pvy, pvp.argmax(1)), "PV test F1": macro_f1(pvy, pvp.argmax(1)),
             "PD test acc": 100 * accuracy_score(pdy, pdp.argmax(1)), "PD test F1": macro_f1(pdy, pdp.argmax(1))}
        lo, hi = bootstrap_ci(pdy.numpy(), pdp.argmax(1).numpy()); r["PD F1 95% CI"] = f"[{lo:.1f}, {hi:.1f}]"
        p2_rows.append(r); p2_preds[(seed, tta)] = pdp.argmax(1).numpy()
    if seed == P2_SEEDS[0]:
        torch.save({"model_state": m.state_dict(), "class_names": class_names, "backbone": best_backbone, "epoch": ep},
                   OUT / f"{best_backbone.lower().replace('-', '_')}_joint.pt")
        y_pd_test = pdy.numpy()
    print(f"seed {seed}: selected epoch {ep}, {time.time() - t0:.0f}s", flush=True)
    del m
    if fast.type == "mps": torch.mps.empty_cache()

p2 = pd.DataFrame(p2_rows); p2.to_csv(OUT / "p2_joint_finetuning_runs.csv", index=False)
display(p2.round(2))
p2_summary = p2.groupby("TTA")[["PV test acc", "PV test F1", "PD test acc", "PD test F1"]].agg(["mean", "std"])
p2_summary.to_csv(OUT / "p2_summary.csv"); display(p2_summary.round(2))

  Swin-T seed 0 epoch 1: loss 1.436 | PV val 93.13 | field val 61.54


  Swin-T seed 0 epoch 2: loss 1.029 | PV val 95.39 | field val 65.19


  Swin-T seed 0 epoch 3: loss 0.924 | PV val 97.91 | field val 67.83


  Swin-T seed 0 epoch 4: loss 0.855 | PV val 97.57 | field val 71.03


  Swin-T seed 0 epoch 5: loss 0.809 | PV val 98.41 | field val 66.49


  Swin-T seed 0 epoch 6: loss 0.774 | PV val 98.53 | field val 69.63


... (31 outputs left out)

  Swin-T seed 2 epoch 12: loss 0.679 | PV val 99.39 | field val 71.26


seed 2: selected epoch 7, 1122s


,backbone,seed,TTA,epoch,PV val F1,field val F1,PV test acc,PV test F1,PD test acc,PD test F1,PD F1 95% CI
0,Swin-T,0,False,7,98.46,74.63,98.37,98.54,67.68,68.59,"[59.0, 75.8]"
1,Swin-T,0,True,7,98.46,74.63,98.37,98.59,66.67,67.91,"[58.2, 75.2]"
2,Swin-T,1,False,7,98.57,72.96,97.86,97.94,71.72,72.41,"[62.8, 79.7]"
3,Swin-T,1,True,7,98.57,72.96,98.11,98.21,68.69,70.15,"[60.3, 77.3]"
4,Swin-T,2,False,7,98.90,73.09,98.40,98.62,71.72,72.14,"[62.5, 79.2]"
5,Swin-T,2,True,7,98.90,73.09,98.37,98.56,69.70,70.47,"[60.9, 77.3]"


PV test acc       PV test F1       PD test acc       PD test F1      
             mean   std       mean   std        mean   std       mean   std
TTA                                                                        
False       98.21  0.30      98.37  0.38       70.37  2.33      71.05  2.13
True        98.28  0.15      98.45  0.21       68.35  1.54      69.51  1.40

In [12]:
p, r, f, s = precision_recall_fscore_support(y_pd_test, p2_preds[(P2_SEEDS[0], True)], labels=range(K), zero_division=0)
p2_per_class = pd.DataFrame({"Class": class_names, "Precision": p, "Recall": r, "F1": f, "Test images": s})
p2_per_class.to_csv(OUT / "p2_plantdoc_per_class.csv", index=False); display(p2_per_class.round(3))
pd.DataFrame({"path": [str(x) for x in test_frames["PlantDoc test"]["path"]], "true": [class_names[i] for i in y_pd_test],
              "pred_P2": [class_names[i] for i in p2_preds[(P2_SEEDS[0], True)]]}).to_csv(OUT / "p2_plantdoc_test_predictions.csv", index=False)
json.dump({"best_backbone": best_backbone, "p2_epochs": P2_EPOCHS, "p2_seeds": P2_SEEDS, "smoke": SMOKE, "device": str(fast)},
          open(OUT / "p2_run_info.json", "w"), indent=2)
print("Part 2 outputs saved to", OUT)

,Class,Precision,Recall,F1,Test images
0,Apple___Apple_scab,1.000,0.900,0.947,10
1,Corn_(maize)___Cercospora_leaf_spot Gray_leaf_...,1.000,1.000,1.000,4
2,Grape___Black_rot,1.000,1.000,1.000,8
3,Potato___Early_blight,0.667,0.250,0.364,8
4,Potato___Late_blight,0.400,0.750,0.522,8
5,Squash___Powdery_mildew,1.000,1.000,1.000,6
6,Tomato___Bacterial_spot,0.571,0.444,0.500,9
7,Tomato___Early_blight,0.778,0.778,0.778,9
8,Tomato___Late_blight,0.600,0.600,0.600,10
9,Tomato___Leaf_Mold,0.333,0.333,0.333,6


Part 2 outputs saved to ~/Desktop/Plant_Disease_Domain_Gap/repo/outputs/part2
